# HealthCost AI — Preprocessing Pipeline

**Goal:** Build a single, reusable sklearn Pipeline (ColumnTransformer + imputation/encoding/scaling) 
that will later be combined with the model and saved as one joblib artifact — the same pipeline 
used for both training and inference (no manual preprocessing duplicated in the API).

**Input:** data/raw/insurance.csv (cleaned in notebook 01: duplicates removed, 1337 rows)
**Output:** train/test split saved to data/processed/, fitted ColumnTransformer ready for modeling notebook

## Load Cleaned Data & Train/Test Split
Splitting before any fitting to prevent data leakage.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/raw/medical_insurance.csv')
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=['charges'])
y = np.log1p(df['charges'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape)
X_train.head()

(1069, 6) (268, 6)


,age,sex,bmi,children,smoker,region
1113,23,male,24.510,0,no,northeast
967,21,male,25.745,2,no,northeast
598,52,female,37.525,2,no,northwest
170,63,male,41.470,0,no,southeast
275,47,female,26.600,2,no,northeast


## Build ColumnTransformer
Numeric features scaled; categorical features one-hot encoded.
Includes imputers for robustness even though this dataset has no missing values.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train_transformed = preprocessor.fit_transform(X_train)
print(X_train_transformed.shape)
print(preprocessor.get_feature_names_out())

(1069, 11)
['num__age' 'num__bmi' 'num__children' 'cat__sex_female' 'cat__sex_male'
 'cat__smoker_no' 'cat__smoker_yes' 'cat__region_northeast'
 'cat__region_northwest' 'cat__region_southeast' 'cat__region_southwest']


## Save Train/Test Split
Persisting the split (before transformation) to data/processed/, so modeling notebook 
loads the exact same split — the ColumnTransformer itself will be fit again inside the 
full pipeline in the modeling notebook (not saved separately here) to keep everything 
bundled as one artifact later.

In [4]:
import os

os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Saved to data/processed/")

Saved to data/processed/


## Summary

**What this notebook did:**
- Loaded cleaned data (duplicates removed), split into train/test (1069 / 268 rows, 80/20, random_state=42)
- Target transformed with log1p (decision from notebook 01), inverse with expm1 at inference time
- Built and tested a ColumnTransformer: StandardScaler for numeric (age, bmi, children), 
  OneHotEncoder for categorical (sex, smoker, region) — both wrapped with imputers for robustness
- Verified transform output: 11 columns after encoding (as expected: 3 numeric + 2 sex + 2 smoker + 4 region)
- Saved train/test split (untransformed) to data/processed/

**Decision carried forward:**
- The ColumnTransformer built here was only a validation test — it is NOT saved separately.
- In the modeling notebook, this same ColumnTransformer logic will be re-defined inside a single 
  sklearn Pipeline together with the final model, fit once, and saved as one joblib artifact 
  (models/final_model.joblib). This avoids any train/inference preprocessing mismatch.

**Next:** modeling notebook — baseline Linear Regression → Random Forest → (if justified) Gradient 
Boosting/XGBoost, evaluated on the held-out test set with MAE/RMSE/R².